# Busy Buffet — Phase 2 Data Analysis

> **Presented by** Yanisa Mahuppon

---
## Step 0 — Import Libraries and Load Data

In [37]:
import pandas as pd
import numpy as np
from datetime import timedelta
import plotly.graph_objects as go
import plotly.express as px

print('Ready')

Ready


In [38]:
TEMPLATE = 'plotly_white'

C_INHOUSE  = '#1B365D'  
C_WALKIN   = '#5B9BD5'  
C_CREAM    = '#E7E6E6'  
C_GRAY     = '#A5A5A5'  

C_GREEN    = "#92DE76"  
C_RED      = "#CF766C"  

COLOR_SCALE = [C_INHOUSE, C_WALKIN, '#8497B0', '#ADB9CA', C_CREAM]

In [39]:
df = pd.read_pickle('../data/busy_buffet_clean.pkl')

print(f'โหลดสำเร็จ — {len(df):,} แถว')

โหลดสำเร็จ — 363 แถว


สร้าง occupancy data (pax ต่อ 30-min slot) ไว้ใช้ใน Comment 2

In [40]:
slots  = [timedelta(hours=h, minutes=m) for h in range(6, 14) for m in (0, 30)]
seated = df[df['seating_status'] != 'Walk-away']

occ_rows = []
for _, row in seated.iterrows():
    if pd.isna(row['meal_start']) or pd.isna(row['meal_end']):
        continue
    pax_val = row['pax'] if row['pax'] > 0 else 1
    for slot in slots:
        if row['meal_start'] <= slot < row['meal_end']:
            h = slot.seconds // 3600
            m = (slot.seconds % 3600) // 60
            occ_rows.append({
                'date'    : row['date'],
                'slot_str': f'{h:02d}:{m:02d}',
                'pax'     : pax_val,
            })

occ_df  = pd.DataFrame(occ_rows)
n_days  = df['date'].nunique()
occ_avg = occ_df.groupby('slot_str')['pax'].sum().div(n_days).round(1).reset_index()
occ_avg.columns = ['slot', 'avg_pax']

print('Occupancy data พร้อม')


Occupancy data พร้อม


---
# Task 1 — For each staff comment, create visuals and analysis to prove if their statement is true or not
---
## Comment 1
>“In-house (hotel) customers are unhappy that they have to wait for a table.
Walk-in customers are also unhappy, when they queue up for a long time
and leave the queue because they don’t want to wait any longer”

**Chart A — In-house Experience Gap**

ความต่างของเวลารอคอยระหว่างลูกค้าพักในโรงแรมและ Walk-in

In [41]:
# เตรียมข้อมูล: เวลารอเฉลี่ย แยก guest type
wait_avg = (
    df[df['wait_time_min'] > 0]
    .groupby('Guest_type')['wait_time_min']
    .mean()
    .round(1)
    .reset_index()
)
wait_avg.columns = ['Guest_type', 'avg_wait']
wait_avg

,Guest_type,avg_wait
0,In house,28.0
1,Walk in,38.4


In [42]:
fig = go.Figure(go.Bar(
    x=wait_avg['Guest_type'],
    y=wait_avg['avg_wait'],
    marker_color=[C_INHOUSE, C_WALKIN],
    text=wait_avg['avg_wait'].apply(lambda v: f'{v:.0f} นาที'),
    textposition='outside',
    width=0.4,
))

fig.update_layout(
    title='Chart A — In-house Experience Gap',
    yaxis_title='นาที',
    yaxis_range=[0, 55],
    template=TEMPLATE,
    showlegend=False,
)
fig.show()

**Chart B — Loss of Opportunity**

จำนวนกลุ่มลูกค้าที่ยกเลิกการรอเนื่องจากระยะเวลารอนานเกินไป

In [43]:
# เตรียมข้อมูล: จำนวน walk-away
walkaway_count = (
    df[df['is_walkaway']]
    .groupby('Guest_type')
    .size()
    .reset_index(name='walk_away')
)
walkaway_count

,Guest_type,walk_away
0,In house,7
1,Walk in,7


In [44]:
fig = go.Figure(go.Bar(
    x=walkaway_count['Guest_type'],
    y=walkaway_count['walk_away'],
    marker_color=[C_INHOUSE, C_WALKIN],
    text=walkaway_count['walk_away'],
    textposition='outside',
    width=0.4,
))

fig.update_layout(
    title='Chart B — Loss of Opportunity',
    yaxis_title='จำนวน groups',
    yaxis_range=[0, 11],
    template=TEMPLATE,
    showlegend=False,
)
fig.show()

## Comment 1
>“In-house (hotel) customers are unhappy that they have to wait for a table.
Walk-in customers are also unhappy, when they queue up for a long time
and leave the queue because they don’t want to wait any longer”


**Insight :** ลูกค้า Walk-in รอนานเฉลี่ยถึง 38 นาที (มากกว่า In-house 10 นาที) และพบว่ามีการ "ทิ้งคิว" (Walk-away) เกิดขึ้นจริงทั้งสองกลุ่ม

**Conclusion :** Comment 1 เป็นจริง ความล่าช้าในการจัดสรรโต๊ะส่งผลกระทบต่อความพึงพอใจและสูญเสียโอกาสในการขาย


---
## Comment 2
> “We are very busy every day of the week. If it’s going to be this busy every
week I think it’s impossible to sustain this business. This buffet business is not
possible for this hotel”

**Chart A — Daily Demand Consistency**

 ปริมาณลูกค้าที่เข้าใช้บริการอย่างต่อเนื่องรายวัน

In [45]:
# เตรียมข้อมูล: pax รายวัน
daily_pax = (
    df[df['seating_status'] != 'Walk-away']
    .groupby('date')['pax']
    .sum()
    .reset_index()
)
daily_pax['date_str'] = daily_pax['date'].dt.strftime('%d %b\n(%a)')
daily_pax

,date,pax,date_str
0,2026-03-13,102.0,13 Mar\n(Fri)
1,2026-03-14,153.0,14 Mar\n(Sat)
2,2026-03-15,151.0,15 Mar\n(Sun)
3,2026-03-17,118.0,17 Mar\n(Tue)
4,2026-03-18,122.0,18 Mar\n(Wed)


In [46]:
fig = go.Figure(go.Bar(
    x=daily_pax['date_str'],
    y=daily_pax['pax'],
    marker_color=C_GRAY,
    text=daily_pax['pax'].astype(int),
    textposition='outside',
    width=0.5,
))

fig.update_layout(
    title='Chart A — Daily Demand Consistency',
    yaxis_title='จำนวน pax',
    yaxis_range=[0, 200],
    template=TEMPLATE,
    showlegend=False,
)
fig.show()

**Chart B — Peak Load Identification**

การวิเคราะห์ช่วงเวลาที่มีความหนาแน่นสูงสุด

In [47]:
fig = go.Figure(go.Bar(
    x=occ_avg['slot'],
    y=occ_avg['avg_pax'],
    marker_color=[
        C_INHOUSE if v >= 25 else C_WALKIN if v >= 10 else C_CREAM
        for v in occ_avg['avg_pax']
    ],
    text=occ_avg['avg_pax'],
    textposition='outside',
))

fig.update_layout(
    title='Chart B — Peak Load Identification',
    yaxis_title='pax เฉลี่ย (ต่อวัน)',
    xaxis_title='เวลา',
    yaxis_range=[0, 42],
    template=TEMPLATE,
    showlegend=False,
)
fig.show()

## Comment 2
> “We are very busy every day of the week. If it’s going to be this busy every
week I think it’s impossible to sustain this business. This buffet business is not
possible for this hotel”

**Insight :** ปริมาณลูกค้าหนาแน่นทุกวัน แต่ความแออัดไม่ได้เกิดตลอดเวลา โดยจะพีคสูงสุดในช่วง 08:30–10:30 น.

**Conclusion :** Comment 2 เป็นจริงบางส่วน ปัญหาไม่ได้อยู่ที่การจัดการตลอดทั้งวัน แต่อยู่ที่การรับมือกับ "Peak Hour" ที่ชัดเจน


---
## Comment 3
> “Walk-in customers sit the whole day. It's very difficult to find seats for
in-house customers. We don’t have enough tables so when one customer sits
for a long time it makes the queue very long”

**Chart A — Turnover Analysis**

เปรียบเทียบพฤติกรรมการใช้โต๊ะระหว่างลูกค้าสองกลุ่ม

In [48]:
# เตรียมข้อมูล: meal duration เฉลี่ย
dur_avg = (
    df.groupby('Guest_type')['meal_duration_min']
    .mean()
    .round(1)
    .reset_index()
)
dur_avg.columns = ['Guest_type', 'avg_dur']
dur_avg

,Guest_type,avg_dur
0,In house,45.5
1,Walk in,72.8


In [49]:
fig = go.Figure(go.Bar(
    x=dur_avg['Guest_type'],
    y=dur_avg['avg_dur'],
    marker_color=[C_INHOUSE, C_WALKIN],
    text=dur_avg['avg_dur'].apply(lambda v: f'{v:.0f} นาที'),
    textposition='outside',
    width=0.4,
))

fig.update_layout(
    title='Chart A — Turnover Analysis',
    yaxis_title='นาที',
    yaxis_range=[0, 95],
    template=TEMPLATE,
    showlegend=False,
)
fig.show()

In [50]:
# เตรียมข้อมูล: % แต่ละกลุ่มที่นั่งในแต่ละช่วง
bins   = [0, 60, 120, 180, 999]
labels = ['< 1 ชั่วโมง', '1–2 ชั่วโมง', '2–3 ชั่วโมง', '3+ ชั่วโมง']

df['dur_group'] = pd.cut(df['meal_duration_min'], bins=bins, labels=labels)

dur_pct = (
    df.groupby(['Guest_type', 'dur_group'], observed=True)
    .size()
    .reset_index(name='n')
)
total = dur_pct.groupby('Guest_type')['n'].transform('sum')
dur_pct['pct'] = (dur_pct['n'] / total * 100).round(1)
dur_pct

,Guest_type,dur_group,n,pct
0,In house,< 1 ชั่วโมง,125,83.9
1,In house,1–2 ชั่วโมง,19,12.8
2,In house,2–3 ชั่วโมง,3,2.0
3,In house,3+ ชั่วโมง,2,1.3
4,Walk in,< 1 ชั่วโมง,87,43.7
5,Walk in,1–2 ชั่วโมง,91,45.7
6,Walk in,2–3 ชั่วโมง,19,9.5
7,Walk in,3+ ชั่วโมง,2,1.0


**Chart B — Seating Efficiency**

สัดส่วนระยะเวลาการใช้บริการเพื่อประเมินรอบการหมุนเวียนโต๊ะ

In [51]:
bucket_colors = ['#1B365D', '#4671A8', '#8497B0', '#ADB9CA', '#E7E6E6']

fig = px.bar(
    dur_pct,
    x='Guest_type',
    y='pct',
    color='dur_group',
    color_discrete_sequence=bucket_colors,
    barmode='stack',
    text='pct',
    template=TEMPLATE,
    labels={'pct': '%', 'Guest_type': '', 'dur_group': 'ช่วงเวลา'},
    title='Chart B — Seating Efficiency',
)

fig.update_traces(
    texttemplate='%{text:.0f}%',
    textposition='inside',
)
fig.update_layout(
    yaxis_ticksuffix='%',
    yaxis_range=[0, 110],
)
fig.show()

## Comment 3
> “Walk-in customers sit the whole day. It's very difficult to find seats for
in-house customers. We don’t have enough tables so when one customer sits
for a long time it makes the queue very long”

**Insight :** ลูกค้า Walk-in นั่งนานเฉลี่ย 73 นาที ในขณะที่ In-house ใช้เวลาเพียง 46 นาที โดย In-house กว่า 84% ทานเสร็จภายใน 1 ชม.

**Conclusion :** Comment 3 เป็นจริง พฤติกรรมการนั่งนานของ Walk-in ส่งผลกระทบโดยตรงต่อรอบการหมุนเวียนโต๊ะ (Table Turnover) สำหรับรองรับแขกโรงแรม


---
# Task 2 — For each of the recommended actions, create visual and analysis to disprove why each of them will not work.

---
## Action 1 
>Reduce seating time (5 hours to less)


**Chart Action 1 — Policy Impact Assessment**

การประเมินผลกระทบต่อลูกค้าหากมีการจำกัดเวลาใช้บริการ

In [52]:
# % ที่นั่งเกิน 2h แยก guest type
rows = []
for guest in ['In house', 'Walk in']:
    sub   = df[df['Guest_type'] == guest]['meal_duration_min'].dropna()
    over  = round((sub > 120).sum() / len(sub) * 100, 1)
    under = round(100 - over, 1)
    rows.append({'Guest_type': guest, 'นั่งไม่ถึง 2h': under, 'นั่งเกิน 2h': over})

a1_df = pd.DataFrame(rows)
a1_df

,Guest_type,นั่งไม่ถึง 2h,นั่งเกิน 2h
0,In house,96.7,3.3
1,Walk in,89.4,10.6


In [53]:
a1_melt = a1_df.melt(id_vars='Guest_type', var_name='กลุ่ม', value_name='pct')

fig = px.bar(
    a1_melt,
    x='Guest_type',
    y='pct',
    color='กลุ่ม',
    color_discrete_map={'นั่งไม่ถึง 2h': C_INHOUSE, 'นั่งเกิน 2h': C_RED},
    barmode='stack',
    text='pct',
    template=TEMPLATE,
    labels={'pct': '%', 'Guest_type': ''},
    title='Chart Action 1 — Policy Impact Assessment',
)

fig.update_traces(
    texttemplate='%{text:.0f}%',
    textposition='inside',
)
fig.update_layout(
    yaxis_ticksuffix='%',
    yaxis_range=[0, 110],
)
fig.show()

## Action 1 
>Reduce seating time (5 hours to less)

**Insight :** ลูกค้า In-house ถึง 97% ใช้เวลาไม่เกิน 2 ชม. อยู่แล้ว การประกาศจำกัดเวลาจึงไม่ช่วยลดความแออัด

**Conclusion :** Action 1 ไม่ได้ผล เพราะไม่สามารถแก้ปัญหาที่ต้นเหตุ แต่จะสร้างภาพลักษณ์ที่เป็นลบต่อโรงแรมแทน


---
## Action 2 
>Increase price everyday to 259



**Chart Action 2 (A) — Revenue Contribution Mix**
โครงสร้างกลุ่มลูกค้าเพื่อการตัดสินใจด้านกลยุทธ์ราคา

In [54]:
# สัดส่วน guest type รวม
guest_total = df['Guest_type'].value_counts().reset_index()
guest_total.columns = ['Guest_type', 'count']
guest_total

,Guest_type,count
0,Walk in,206
1,In house,157


In [55]:
fig = px.pie(
    guest_total,
    names='Guest_type',
    values='count',
    color='Guest_type',
    color_discrete_map={'In house': C_INHOUSE, 'Walk in': C_WALKIN},
    hole=0.5,
    template=TEMPLATE,
    title='Chart Action 2 (A) — Revenue Contribution Mix',
)

fig.update_traces(
    textinfo='label+percent',
    textposition='outside',
)
fig.show()

**Chart Action 2 (B) — Daily Segment Stability**

ความเสถียรของสัดส่วนลูกค้า Walk-in ซึ่งเป็นรายได้หลัก

In [56]:
# guest mix รายวัน
guest_daily = (
    df.groupby(['date', 'Guest_type'])
    .size()
    .reset_index(name='count')
)
guest_daily['date_str'] = guest_daily['date'].dt.strftime('%d %b')

In [57]:
fig = px.bar(
    guest_daily,
    x='date_str',
    y='count',
    color='Guest_type',
    color_discrete_map={'In house': C_INHOUSE, 'Walk in': C_WALKIN},
    barmode='group',
    text='count',
    template=TEMPLATE,
    labels={'count': 'จำนวน groups', 'date_str': 'วันที่', 'Guest_type': ''},
    title='Chart Action 2 (B) — Daily Segment Stability',
)

fig.update_traces(textposition='outside')
fig.show()

## Action 2 
>Increase price everyday to 259

**Insight :** ลูกค้า Walk-in คิดเป็น 57% ของกลุ่มผู้ใช้บริการทั้งหมด และเป็นกลุ่มหลักที่สร้างรายได้เงินสด

**Conclusion :** Action 2 ไม่ได้ผล การขึ้นราคาอาจทำให้เสียฐานรายได้หลัก โดยที่กลุ่ม In-house (ซึ่งจ่ายรวมมากับค่าห้อง) ก็ยังคงประสบปัญหาไม่มีที่นั่งอยู่ดี


---
## Action 3 
>Queue skipping for in-house guest



**Chart Action 3 — Supply-Demand Imbalance**

การเปรียบเทียบขีดความสามารถในการรองรับลูกค้า (Capacity) กับความต้องการจริงในช่วง Peak

In [58]:
peak_start = timedelta(hours=8)
peak_end   = timedelta(hours=10)

wi_seated  = df[(df['Guest_type'] == 'Walk in') & (df['seating_status'] != 'Walk-away') & df['meal_start'].notna() & df['meal_end'].notna()].copy()
wi_in_peak = wi_seated[(wi_seated['meal_start'] < peak_end) & (wi_seated['meal_end'] > peak_start)]

ih_queued  = df[(df['Guest_type'] == 'In house') & (df['seating_status'] == 'Queued')].copy()
hq = ih_queued['queue_start'].notna() & ih_queued['queue_end'].notna()
ih_queued['wait'] = np.where(hq, (ih_queued['queue_end'] - ih_queued['queue_start']).dt.total_seconds() / 60, np.nan)

avg_wi_dur  = wi_in_peak['meal_duration_min'].mean().round(0)
avg_ih_wait = ih_queued['wait'].mean().round(0)

print(f'Walk-in นั่งอยู่ช่วง peak เฉลี่ย : {avg_wi_dur:.0f} นาที')
print(f'In-house รอคิวเฉลี่ย  : {avg_ih_wait:.0f} นาที')

Walk-in นั่งอยู่ช่วง peak เฉลี่ย : 81 นาที
In-house รอคิวเฉลี่ย  : 28 นาที


In [59]:
a3_df = pd.DataFrame({
    'กลุ่ม'    : ['Walk-in นั่งอยู่ช่วง peak', 'In-house รอคิว'],
    'นาที'     : [avg_wi_dur, avg_ih_wait],
    'color'    : [C_WALKIN, C_INHOUSE],
})

fig = go.Figure(go.Bar(
    x=a3_df['กลุ่ม'],
    y=a3_df['นาที'],
    marker_color=a3_df['color'],
    text=a3_df['นาที'].apply(lambda v: f'{v:.0f} นาที'),
    textposition='outside',
    width=0.4,
))

fig.update_layout(
    title='Chart Action 3 — Supply-Demand Imbalance',
    yaxis_title='นาที',
    yaxis_range=[0, 110],
    template=TEMPLATE,
    showlegend=False,
)
fig.show()

## Action 3 
>Queue skipping for in-house guest

**Insight :** In-house รอคิวเฉลี่ย 28 นาที แต่ walk-in ที่นั่งอยู่ช่วงเดียวกัน
ใช้เวลาเฉลี่ย 81 นาที Queue skip ช่วยลำดับได้ แต่โต๊ะยังไม่ว่างอยู่ดีตราบใดที่ walk-in นั่งนาน

**Conclusion :** Action 3 ไม่ได้ผล เพราะปัญหาคือ "โต๊ะเต็ม" (Capacity) ไม่ใช่ "ลำดับคิว" ต่อให้ข้ามคิวมาได้ก็ไม่มีโต๊ะให้นั่ง


---
# Task 3 — Reserved Seating for In-house

---

ปรับจาก Queue Skipping → **จัดโต๊ะ Reserved Zone** แยกสำหรับ in-house  
In-house แจ้ง preferred time ผ่าน front desk ตอน check-in

**Chart A — Preferred Arrival Patterns**

ช่วงเวลาเข้าใช้บริการยอดนิยมของกลุ่มลูกค้า In-house เพื่อการจัดสรรพื้นที่

In [60]:
# เวลาที่ in-house นิยมมากิน
ih_seated = df[(df['Guest_type'] == 'In house') & df['meal_start'].notna()].copy()
ih_seated['meal_hour'] = ih_seated['meal_start'].apply(
    lambda x: x.seconds // 3600 if pd.notna(x) else None
)

ih_hour = (
    ih_seated[ih_seated['meal_hour'].between(6, 13)]
    .groupby('meal_hour')
    .size()
    .reset_index(name='count')
)
ih_hour['hour_str'] = ih_hour['meal_hour'].apply(lambda h: f'{h:02d}:00')

In [61]:
fig = go.Figure(go.Bar(
    x=ih_hour['hour_str'],
    y=ih_hour['count'],
    marker_color=[C_INHOUSE if h in [7, 8, 9] else C_WALKIN for h in ih_hour['meal_hour']],
    text=ih_hour['count'],
    textposition='outside',
))

fig.update_layout(
    title='Chart A — Preferred Arrival Patterns',
    yaxis_title='จำนวน groups (รวม 5 วัน)',
    xaxis_title='เวลา',
    yaxis_range=[0, 80],
    template=TEMPLATE,
    showlegend=False,
)
fig.show()

**Chart B — Optimized Inventory Allocation**

การกำหนดจำนวนโต๊ะสำรองที่เหมาะสมตามความต้องการจริงรายวัน

In [62]:
# จำนวนโต๊ะที่ต้อง reserve รายวัน
ih_daily = (
    df[df['Guest_type'] == 'In house']
    .groupby('date')
    .agg(pax=('pax', 'sum'))
    .reset_index()
)
ih_daily['date_str']      = ih_daily['date'].dt.strftime('%d %b')
ih_daily['tables_needed'] = (ih_daily['pax'] / 2).apply(np.ceil).astype(int)
ih_daily[['date_str', 'pax', 'tables_needed']]

,date_str,pax,tables_needed
0,13 Mar,53.0,27
1,14 Mar,53.0,27
2,15 Mar,51.0,26
3,17 Mar,56.0,28
4,18 Mar,62.0,31


In [63]:
avg_t = ih_daily['tables_needed'].mean()

fig = go.Figure()

fig.add_trace(go.Bar(
    x=ih_daily['date_str'],
    y=ih_daily['tables_needed'],
    marker_color=C_GRAY,
    text=ih_daily['tables_needed'],
    textposition='outside',
    width=0.5,
    name='โต๊ะที่ต้องการ',
))

fig.add_hline(
    y=avg_t,
    line_dash='dash',
    line_color='navy',
    annotation_text=f'เฉลี่ย {avg_t:.0f} โต๊ะ/วัน',
    annotation_position='top right',
)

fig.update_layout(
    title='Chart B — Optimized Inventory Allocation',
    yaxis_title='จำนวนโต๊ะ',
    yaxis_range=[0, 50],
    template=TEMPLATE,
    showlegend=False,
)
fig.show()

# Task 3 — Reserved Seating for In-house


**Insight :** ข้อมูลระบุว่า In-house มักเข้าใช้บริการช่วง 07:00–09:00 น. ซึ่งเป็นช่วงก่อนที่ Walk-in จะเข้ามาหนาแน่น

**Proposed Implementation :** ควรสำรองโต๊ะไว้ประมาณ 28 โต๊ะ (ตามค่าเฉลี่ยความต้องการจริง) ให้กลุ่ม In-house ในช่วงเวลาดังกล่าว

**Why it works :**

* Guaranteed Experience: แขกโรงแรมมั่นใจว่ามีที่นั่งแน่นอน

* Revenue Control: ไม่เสียรายได้จาก Walk-in เพราะสามารถคืนโต๊ะ (Release) ให้ Walk-in ได้ทันทีหลัง 09:00 น.

* Proactive Management: โรงแรมสามารถจัดสรรทรัพยากรบุคคลได้ตามการจองของแขก

---
## สรุป Phase 2 — Data Analysis

| Task | หัวข้อ | Verdict | สรุปสำคัญ |
|------|--------|---------|-----------|
| Comment 1 | In-house รอนาน, Walk-in หนี | ✅ TRUE | Walk-in รอเฉลี่ย 38 นาที vs In-house 28 นาที มี walk-away ทั้ง 2 กลุ่ม |
| Comment 2 | ยุ่งทุกวัน บริหารไม่ไหว | ✅ TRUE (บางส่วน) | ยุ่งทุกวัน แต่ peak อยู่แค่ช่วง 08:30–10:30 ไม่ใช่ตลอดวัน |
| Comment 3 | Walk-in นั่งทั้งวัน หาโต๊ะให้ In-house ไม่ได้ | ✅ TRUE | Walk-in นั่งเฉลี่ย 73 นาที vs In-house 46 นาที (เกือบ 2 เท่า) |
| Action 1 | ลดเวลาบุฟเฟ่ต์จาก 5h เป็นน้อยกว่า | ❌ ไม่ได้ผล | In-house 97% นั่งไม่ถึง 2h อยู่แล้ว ลด limit ไม่มีผล แต่สร้างภาพลักษณ์ที่เป็นลบ |
| Action 2 | ขึ้นราคาเป็น 259 บาท ทุกวัน | ❌ ไม่ได้ผล | Walk-in = 57% ของลูกค้า + In-house ส่วนใหญ่ได้ breakfast รวมในห้องอยู่แล้ว ขึ้นราคาเสียรายได้ |
| Action 3 | Queue Skipping สำหรับ In-house | ❌ ไม่ได้ผล | Walk-in นั่งอยู่ช่วง peak เฉลี่ย 81 นาที — ปัญหาคือโต๊ะเต็ม ไม่ใช่ลำดับคิว |
| Task 3 | **แนะนำ: Reserved Seating Zone สำหรับ In-house** | ✅ แก้ปัญหาได้จริง | Reserve ~28 โต๊ะ ช่วง 07:00–09:00 / คืนโต๊ะให้ Walk-in ทันทีหลัง 09:00 ไม่กระทบรายได้ |